# Setup do ambiente de treino

Confere GPU/CUDA disponíveis, carrega o checkpoint base do YOLO26m e valida que o dataset processado (gerado por `src/preprocessing/pipeline.py`) está no lugar certo, antes de partir pro treino em `02_train.ipynb`.

In [2]:
import sys
import torch

print(f"Python:     {sys.version}")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.version.cuda}")
print(f"GPU disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  GPU não encontrada, treino será feito em CPU")

Python:     3.12.8 (main, Jan 14 2025, 22:49:36) [MSC v.1942 64 bit (AMD64)]
PyTorch:    2.11.0+cu128
CUDA:       12.8
GPU disponível: True
GPU: NVIDIA GeForce GTX 1650
VRAM: 4.0 GB


Confere a versão do Ultralytics instalada e roda o diagnóstico de ambiente da própria lib.

In [2]:
import ultralytics

print(f"Ultralytics: {ultralytics.__version__}")
ultralytics.checks()

Ultralytics 8.4.38  Python-3.12.8 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Setup complete  (12 CPUs, 15.8 GB RAM, 216.4/237.8 GB disk)


Baixa (ou carrega, se já existir) o checkpoint pré-treinado `yolo26m.pt` — ponto de partida do fine-tuning, não é o modelo final do TCC1 (esse fica em `runs/baseline/`, usado como referência no `03_eval.ipynb`).

In [3]:
from ultralytics import YOLO
from pathlib import Path
import shutil

model_path = Path("../runs/yolo26m.pt")

if not model_path.exists():
    model = YOLO("yolo26m.pt")
    shutil.move("yolo26m.pt", model_path)
else:
    model = YOLO(model_path)

print(f"Modelo carregado de: {model_path}")

Modelo carregado de: ..\runs\yolo26m.pt


Confere que as pastas do dataset processado existem e têm arquivo. Se algum item der ❌, rode o pipeline de pré-processamento primeiro (`src/preprocessing/pipeline.py` ou o notebook `03_pipeline_run.ipynb`).

In [4]:
from pathlib import Path

dataset_root = Path("../../../../data/processed/24_chromosomes_object")

paths = {
    "train images": dataset_root / "images/train",
    "val images":   dataset_root / "images/val",
    "train labels": dataset_root / "labels/train",
    "val labels":   dataset_root / "labels/val",
}

for name, path in paths.items():
    exists = path.exists()
    count = len(list(path.iterdir())) if exists else 0
    status = "🟢" if exists else "❌"
    print(f"{status} {name}: {path} ({count} arquivos)")

🟢 train images: ..\..\..\..\data\processed\24_chromosomes_object\images\train (3744 arquivos)
🟢 val images: ..\..\..\..\data\processed\24_chromosomes_object\images\val (1250 arquivos)
🟢 train labels: ..\..\..\..\data\processed\24_chromosomes_object\labels\train (3744 arquivos)
🟢 val labels: ..\..\..\..\data\processed\24_chromosomes_object\labels\val (1250 arquivos)


Smoke test: roda uma inferência rápida numa imagem de validação, só pra confirmar que o modelo carrega e infere sem erro. Ainda não foi treinado nos cromossomos, então não é esperado detectar nada aqui.

In [5]:

# Pega a primeira imagem do val para teste
val_images = list((dataset_root / "images/val").iterdir())
sample = val_images[0]

results = model(sample)
results[0].show()

print(f"Imagem testada: {sample.name}")
print(f"Detecções: {len(results[0].boxes)}")


image 1/1 c:\Users\irlvinicius\Desktop\CytoML\models\yolo\yolo26m\notebooks\..\..\..\..\data\processed\24_chromosomes_object\images\val\103191.jpg: 640x640 (no detections), 53.7ms
Speed: 10.8ms preprocess, 53.7ms inference, 2.6ms postprocess per image at shape (1, 3, 640, 640)
Imagem testada: 103191.jpg
Detecções: 0
